# 04 — Your first CNN

**Day 1 · Notebook 4 of 4**

We'll train a small CNN on FashionMNIST. The training loop is **fully
explicit** — no Lightning, no `model.fit`. You'll see every step:

`forward → loss → backward → step → zero_grad`.

Target: > 85% test accuracy in a few minutes of CPU training.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms as T
from tqdm.auto import tqdm

from torchinfo import summary

from cvcourse import get_device, count_params, show_grid

# get_device() auto-picks cuda > mps > cpu.
# To force a specific backend, replace with one of:
#   device = torch.device("cpu")
#   device = torch.device("cuda")   # NVIDIA GPU
#   device = torch.device("mps")    # Apple Silicon
device = get_device()
print("device:", device)

## 1. Data


In [ ]:
pipeline = T.Compose([T.ToTensor(), T.Normalize((0.2860,), (0.3530,))])
train_ds = torchvision.datasets.FashionMNIST("../data", train=True,  download=True, transform=pipeline)
test_ds  = torchvision.datasets.FashionMNIST("../data", train=False, download=True, transform=pipeline)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=0)

## 2. Model

Two conv blocks, then a small classifier head. We keep it simple by using
`nn.Sequential` — a straight-line stack of layers, no `forward()` method
needed. Perfect for a first CNN.

Use the shape rule to predict output sizes _before_ you run the cell.


In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),  # 28 -> 14
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),  # 14 -> 7
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)

summary(model, input_size=(1, 1, 28, 28), device=device)  # (batch, C, H, W)

## 3. Sanity-check the forward pass

Always run one batch through the model **before** training. Catches every
shape bug in 5 seconds.


In [ ]:
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
logits = model(xb)
print("input:", xb.shape, "-> logits:", logits.shape)
print("loss on init:", F.cross_entropy(logits, yb).item())

## 4. The training loop

Read each line. No magic.


In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * xb.size(0)
        correct  += (logits.argmax(1) == yb).sum().item()
        total    += xb.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss_sum += loss.item() * xb.size(0)
        correct  += (logits.argmax(1) == yb).sum().item()
        total    += xb.size(0)
    return loss_sum / total, correct / total

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, device)
    print(f"epoch {epoch}  train loss {tr_loss:.3f} acc {tr_acc:.3f}  "
          f"|  test loss {te_loss:.3f} acc {te_acc:.3f}")

## 5. Look at the mistakes

Always inspect failures. They tell you _what_ the model finds hard, which is
much more informative than the aggregate metric.


In [ ]:
model.eval()
xb, yb = next(iter(test_loader))
with torch.no_grad():
    preds = model(xb.to(device)).argmax(1).cpu()
wrong = (preds != yb).nonzero(as_tuple=True)[0][:16]
show_grid(
    xb[wrong],
    titles=[f"{test_ds.classes[yb[i]]}->{test_ds.classes[preds[i]]}" for i in wrong],
    cols=8,
)

## Top-5 first-CNN bugs (memorize)

1. Forgot `.to(device)` for either the model or the batch.
2. Forgot `optimizer.zero_grad()` — gradients accumulate, training looks broken.
3. Forgot `model.eval()` during evaluation — BatchNorm/Dropout misbehave.
4. Wrong shape after `Flatten` — the `Linear` layer size doesn't match.
5. Used the same transform pipeline for train and test, including augmentations.

## You did it

You trained a CNN from scratch. Day 2 picks up here: deeper architectures,
transfer learning, and explainability.

> **Next notebook (05):** we'll rewrite this same model as an `nn.Module`
> subclass so we can branch the forward pass, expose intermediate features,
> and attach hooks (needed for transfer learning and Grad-CAM).
